<a href="https://colab.research.google.com/github/jsl5710/audio_hallucination/blob/claude%2Fcreate-branch-structure-YBUPZ/Hallucination_audio_text_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/ICASSP_Hallucinaton/
%ls /content/drive/MyDrive/ICASSP_Hallucinaton/

/content/drive/MyDrive/ICASSP_Hallucinaton
Audio_data@  checkpoints/  hallucination_results/  tables/


In [ ]:
# Results Table Generator for Hallucination Classification
# Generates LaTeX tables with F1 and Accuracy metrics for Audio and Text experiments
# Includes comparison between Audio and Text experiments

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, classification_report
from typing import Dict, List, Tuple
import json

# ========================== CONFIGURATION ==========================

OUTPUT_DIR = "hallucination_results"
TABLES_DIR = "tables"

MODELS = [
    "Qwen2.5-Omni-3B",
    # "Qwen2.5-Omni-7B"
]

EXPERIMENT_TYPES = ['audio', 'text']
LANGUAGES = ['english', 'kazakh', 'russian']
APPROACHES = ['direct', 'cot']

# Updated type mapping for new classification system
TYPE_MAPPING = {
    'factual_contradiction': 'factual_contradiction',
    'factual_fabrication': 'factual_fabrication',
    'contextual_inconsistency': 'contextual_inconsistency',
    # Backward compatibility
    'contradiction': 'factual_contradiction',
    'fabrication': 'factual_fabrication',
    'none': 'none'
}

# ========================== METRICS CALCULATION ==========================

class MetricsCalculator:
    def __init__(self):
        self.results = {}

    def load_all_results(self) -> Dict:
        """Load all result files for both audio and text experiments"""
        all_results = {}

        for model in MODELS:
            model_dir = Path(OUTPUT_DIR) / model
            if not model_dir.exists():
                print(f"Model directory not found: {model_dir}")
                continue

            all_results[model] = {}

            for experiment_type in EXPERIMENT_TYPES:
                exp_type_dir = model_dir / experiment_type
                if not exp_type_dir.exists():
                    print(f"Experiment type directory not found: {exp_type_dir}")
                    continue

                all_results[model][experiment_type] = {}

                for language in LANGUAGES:
                    result_file = exp_type_dir / f"{language}_results.csv"
                    if result_file.exists():
                        try:
                            df = pd.read_csv(result_file, encoding='utf-8')
                            all_results[model][experiment_type][language] = df
                            print(f"Loaded {model} - {experiment_type} - {language}: {len(df)} samples")
                        except Exception as e:
                            print(f"Failed to load {result_file}: {e}")
                    else:
                        print(f"Result file not found: {result_file}")

        return all_results

    def normalize_types(self, types: List[str]) -> List[str]:
        """Normalize type names using mapping"""
        return [TYPE_MAPPING.get(str(t).lower().strip(), 'none') for t in types]

    def calculate_binary_metrics(self, df: pd.DataFrame, approach: str) -> Tuple[float, float]:
        """Calculate metrics for binary hallucination classification"""
        # Ground truth: all samples should be "yes" since we filtered for hallucination=yes
        y_true = ['yes'] * len(df)
        y_pred = df[f'pred_{approach}_binary'].fillna('no').tolist()

        # Convert to binary (1 for yes, 0 for no)
        y_true_bin = [1 if str(x).lower() == 'yes' else 0 for x in y_true]
        y_pred_bin = [1 if str(x).lower() == 'yes' else 0 for x in y_pred]

        f1 = f1_score(y_true_bin, y_pred_bin, average='binary', zero_division=0)
        acc = accuracy_score(y_true_bin, y_pred_bin)

        return f1, acc

    def calculate_multiclass_metrics(self, df: pd.DataFrame, approach: str, task: str) -> Tuple[float, float]:
        """Calculate metrics for multiclass classification (type or level)"""

        if task == 'type':
            y_true = df['hallucination_type'].fillna('none').tolist()
            y_pred = df[f'pred_{approach}_type'].fillna('none').tolist()

            # Normalize types
            y_true = self.normalize_types(y_true)
            y_pred = self.normalize_types(y_pred)

        elif task == 'level':
            y_true = df['hallucination_level'].fillna('none').tolist()
            y_pred = df[f'pred_{approach}_degree'].fillna('none').tolist()

            # Normalize levels (lowercase, strip)
            y_true = [str(x).lower().strip() for x in y_true]
            y_pred = [str(x).lower().strip() for x in y_pred]
        else:
            raise ValueError(f"Unknown task: {task}")

        # Calculate metrics
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        return f1, acc

    def calculate_all_metrics(self) -> Dict:
        """Calculate all metrics for all models, experiment types, languages, and tasks"""

        all_results = self.load_all_results()
        metrics = {}

        for model, model_data in all_results.items():
            metrics[model] = {}

            for experiment_type, exp_data in model_data.items():
                metrics[model][experiment_type] = {}

                for language, df in exp_data.items():
                    if df.empty:
                        continue

                    metrics[model][experiment_type][language] = {}

                    for approach in APPROACHES:
                        # Check if approach columns exist
                        if f'pred_{approach}_binary' not in df.columns:
                            print(f"Missing {approach} predictions for {model} - {experiment_type} - {language}")
                            continue

                        # Binary classification
                        f1_bin, acc_bin = self.calculate_binary_metrics(df, approach)

                        # Type classification
                        f1_type, acc_type = self.calculate_multiclass_metrics(df, approach, 'type')

                        # Level classification
                        f1_level, acc_level = self.calculate_multiclass_metrics(df, approach, 'level')

                        metrics[model][experiment_type][language][approach] = {
                            'binary': {'f1': f1_bin, 'acc': acc_bin},
                            'type': {'f1': f1_type, 'acc': acc_type},
                            'level': {'f1': f1_level, 'acc': acc_level}
                        }

        self.results = metrics
        return metrics

    def print_metrics_summary(self):
        """Print a readable summary of all metrics"""

        if not self.results:
            self.calculate_all_metrics()

        print("CLASSIFICATION METRICS SUMMARY")
        print("=" * 80)

        for model, model_data in self.results.items():
            print(f"\n{model}")
            print("-" * 60)

            for experiment_type, exp_data in model_data.items():
                print(f"\n{experiment_type.upper()} EXPERIMENTS")
                print("-" * 40)

                for language, lang_data in exp_data.items():
                    print(f"\n{language.upper()}")
                    print("Task              | Approach | F1    | Acc   ")
                    print("-" * 45)

                    for approach in APPROACHES:
                        if approach not in lang_data:
                            continue

                        data = lang_data[approach]

                        # Binary
                        f1_bin = data['binary']['f1']
                        acc_bin = data['binary']['acc']
                        print(f"Binary            | {approach:8} | {f1_bin:.3f} | {acc_bin:.3f}")

                        # Type
                        f1_type = data['type']['f1']
                        acc_type = data['type']['acc']
                        print(f"Type              | {approach:8} | {f1_type:.3f} | {acc_type:.3f}")

                        # Level
                        f1_level = data['level']['f1']
                        acc_level = data['level']['acc']
                        print(f"Level             | {approach:8} | {f1_level:.3f} | {acc_level:.3f}")

# ========================== LATEX TABLE GENERATOR ==========================

class LaTeXTableGenerator:
    def __init__(self, metrics: Dict):
        self.metrics = metrics
        self.tables_dir = Path(TABLES_DIR)
        self.tables_dir.mkdir(exist_ok=True)

    def generate_experiment_type_table(self, model: str, experiment_type: str) -> str:
        """Generate LaTeX table for a specific model and experiment type"""

        if model not in self.metrics or experiment_type not in self.metrics[model]:
            return f"% No data available for {model} - {experiment_type}"

        exp_data = self.metrics[model][experiment_type]

        # Start table
        latex = f"""\\begin{{table*}}[htbp]
\\centering
\\scriptsize
\\renewcommand{{\\arraystretch}}{{1.2}}
\\begin{{tabular}}{{ll|cc|cc}}
\\hline
\\shortstack{{\\textbf{{Classification}} \\\\ \\textbf{{Task}}}} & \\textbf{{Language}}
& \\multicolumn{{2}}{{c|}}{{\\textbf{{Direct}}}}
& \\multicolumn{{2}}{{c}}{{\\textbf{{CoT}}}} \\\\
&
& \\textbf{{F1}} & \\textbf{{Acc}}
& \\textbf{{F1}} & \\textbf{{Acc}} \\\\
\\hline\n"""

        # Binary classification section
        latex += "\\multirow{3}{*}{\\textbf{Binary classification}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            if language in exp_data:
                # Direct approach
                if 'direct' in exp_data[language]:
                    f1_dir = exp_data[language]['direct']['binary']['f1']
                    acc_dir = exp_data[language]['direct']['binary']['acc']
                    direct_f1 = f"{f1_dir:.3f}"
                    direct_acc = f"{acc_dir:.3f}"
                else:
                    direct_f1 = "---"
                    direct_acc = "---"

                # CoT approach
                if 'cot' in exp_data[language]:
                    f1_cot = exp_data[language]['cot']['binary']['f1']
                    acc_cot = exp_data[language]['cot']['binary']['acc']
                    cot_f1 = f"{f1_cot:.3f}"
                    cot_acc = f"{acc_cot:.3f}"
                else:
                    cot_f1 = "---"
                    cot_acc = "---"
            else:
                direct_f1 = direct_acc = cot_f1 = cot_acc = "---"

            latex += f"& {lang_display} & {direct_f1} & {direct_acc} & {cot_f1} & {cot_acc} \\\\\n"

        latex += "\\hline\n\\hline\n"

        # Hallucination type section
        latex += "\\multirow{3}{*}{\\textbf{Hallucination type}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            if language in exp_data:
                # Direct approach
                if 'direct' in exp_data[language]:
                    f1_dir = exp_data[language]['direct']['type']['f1']
                    acc_dir = exp_data[language]['direct']['type']['acc']
                    direct_f1 = f"{f1_dir:.3f}"
                    direct_acc = f"{acc_dir:.3f}"
                else:
                    direct_f1 = "---"
                    direct_acc = "---"

                # CoT approach
                if 'cot' in exp_data[language]:
                    f1_cot = exp_data[language]['cot']['type']['f1']
                    acc_cot = exp_data[language]['cot']['type']['acc']
                    cot_f1 = f"{f1_cot:.3f}"
                    cot_acc = f"{acc_cot:.3f}"
                else:
                    cot_f1 = "---"
                    cot_acc = "---"
            else:
                direct_f1 = direct_acc = cot_f1 = cot_acc = "---"

            latex += f"& {lang_display} & {direct_f1} & {direct_acc} & {cot_f1} & {cot_acc} \\\\\n"

        latex += "\\hline\n"

        # Hallucination level section
        latex += "\\multirow{3}{*}{\\textbf{Hallucination level}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            if language in exp_data:
                # Direct approach
                if 'direct' in exp_data[language]:
                    f1_dir = exp_data[language]['direct']['level']['f1']
                    acc_dir = exp_data[language]['direct']['level']['acc']
                    direct_f1 = f"{f1_dir:.3f}"
                    direct_acc = f"{acc_dir:.3f}"
                else:
                    direct_f1 = "---"
                    direct_acc = "---"

                # CoT approach
                if 'cot' in exp_data[language]:
                    f1_cot = exp_data[language]['cot']['level']['f1']
                    acc_cot = exp_data[language]['cot']['level']['acc']
                    cot_f1 = f"{f1_cot:.3f}"
                    cot_acc = f"{acc_cot:.3f}"
                else:
                    cot_f1 = "---"
                    cot_acc = "---"
            else:
                direct_f1 = direct_acc = cot_f1 = cot_acc = "---"

            latex += f"& {lang_display} & {direct_f1} & {direct_acc} & {cot_f1} & {cot_acc} \\\\\n"

        # End table
        model_safe = model.replace('/', '_').replace('.', '_')
        exp_type_display = experiment_type.capitalize()
        latex += f"""\\hline
\\end{{tabular}}
\\caption{{{exp_type_display} classification performance for {model} (Direct vs Chain-of-Thought)}}
\\label{{tab:hallucination_{model_safe}_{experiment_type}}}
\\end{{table*}}"""

        return latex

    def generate_comparison_table(self, model: str, approach: str) -> str:
        """Generate comparison table between audio and text experiments"""

        if model not in self.metrics:
            return f"% No data available for {model}"

        model_data = self.metrics[model]

        # Start table
        latex = f"""\\begin{{table*}}[htbp]
\\centering
\\scriptsize
\\renewcommand{{\\arraystretch}}{{1.2}}
\\begin{{tabular}}{{ll|cc|cc}}
\\hline
\\shortstack{{\\textbf{{Classification}} \\\\ \\textbf{{Task}}}} & \\textbf{{Language}}
& \\multicolumn{{2}}{{c|}}{{\\textbf{{Audio Input}}}}
& \\multicolumn{{2}}{{c}}{{\\textbf{{Text Input}}}} \\\\
&
& \\textbf{{F1}} & \\textbf{{Acc}}
& \\textbf{{F1}} & \\textbf{{Acc}} \\\\
\\hline\n"""

        approach_display = approach.upper()

        # Binary classification section
        latex += "\\multirow{3}{*}{\\textbf{Binary classification}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            # Audio results
            if 'audio' in model_data and language in model_data['audio'] and approach in model_data['audio'][language]:
                f1_audio = model_data['audio'][language][approach]['binary']['f1']
                acc_audio = model_data['audio'][language][approach]['binary']['acc']
                audio_f1 = f"{f1_audio:.3f}"
                audio_acc = f"{acc_audio:.3f}"
            else:
                audio_f1 = "---"
                audio_acc = "---"

            # Text results
            if 'text' in model_data and language in model_data['text'] and approach in model_data['text'][language]:
                f1_text = model_data['text'][language][approach]['binary']['f1']
                acc_text = model_data['text'][language][approach]['binary']['acc']
                text_f1 = f"{f1_text:.3f}"
                text_acc = f"{acc_text:.3f}"
            else:
                text_f1 = "---"
                text_acc = "---"

            latex += f"& {lang_display} & {audio_f1} & {audio_acc} & {text_f1} & {text_acc} \\\\\n"

        latex += "\\hline\n\\hline\n"

        # Hallucination type section
        latex += "\\multirow{3}{*}{\\textbf{Hallucination type}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            # Audio results
            if 'audio' in model_data and language in model_data['audio'] and approach in model_data['audio'][language]:
                f1_audio = model_data['audio'][language][approach]['type']['f1']
                acc_audio = model_data['audio'][language][approach]['type']['acc']
                audio_f1 = f"{f1_audio:.3f}"
                audio_acc = f"{acc_audio:.3f}"
            else:
                audio_f1 = "---"
                audio_acc = "---"

            # Text results
            if 'text' in model_data and language in model_data['text'] and approach in model_data['text'][language]:
                f1_text = model_data['text'][language][approach]['type']['f1']
                acc_text = model_data['text'][language][approach]['type']['acc']
                text_f1 = f"{f1_text:.3f}"
                text_acc = f"{acc_text:.3f}"
            else:
                text_f1 = "---"
                text_acc = "---"

            latex += f"& {lang_display} & {audio_f1} & {audio_acc} & {text_f1} & {text_acc} \\\\\n"

        latex += "\\hline\n"

        # Hallucination level section
        latex += "\\multirow{3}{*}{\\textbf{Hallucination level}} \n"

        for i, language in enumerate(LANGUAGES):
            lang_display = language.capitalize()

            # Audio results
            if 'audio' in model_data and language in model_data['audio'] and approach in model_data['audio'][language]:
                f1_audio = model_data['audio'][language][approach]['level']['f1']
                acc_audio = model_data['audio'][language][approach]['level']['acc']
                audio_f1 = f"{f1_audio:.3f}"
                audio_acc = f"{acc_audio:.3f}"
            else:
                audio_f1 = "---"
                audio_acc = "---"

            # Text results
            if 'text' in model_data and language in model_data['text'] and approach in model_data['text'][language]:
                f1_text = model_data['text'][language][approach]['level']['f1']
                acc_text = model_data['text'][language][approach]['level']['acc']
                text_f1 = f"{f1_text:.3f}"
                text_acc = f"{acc_text:.3f}"
            else:
                text_f1 = "---"
                text_acc = "---"

            latex += f"& {lang_display} & {audio_f1} & {audio_acc} & {text_f1} & {text_acc} \\\\\n"

        # End table
        model_safe = model.replace('/', '_').replace('.', '_')
        latex += f"""\\hline
\\end{{tabular}}
\\caption{{Audio vs Text input comparison for {model} using {approach_display} approach}}
\\label{{tab:comparison_{model_safe}_{approach}}}
\\end{{table*}}"""

        return latex

    def save_all_tables(self):
        """Generate and save all LaTeX tables"""

        # Individual experiment type tables
        for model in MODELS:
            if model in self.metrics:
                for experiment_type in EXPERIMENT_TYPES:
                    if experiment_type in self.metrics[model]:
                        latex_table = self.generate_experiment_type_table(model, experiment_type)

                        # Save to file
                        model_safe = model.replace('/', '_').replace('.', '_')
                        filename = self.tables_dir / f"table_{model_safe}_{experiment_type}.tex"

                        with open(filename, 'w', encoding='utf-8') as f:
                            f.write(latex_table)

                        print(f"Saved LaTeX table: {filename}")

                # Comparison tables
                for approach in APPROACHES:
                    comparison_table = self.generate_comparison_table(model, approach)

                    model_safe = model.replace('/', '_').replace('.', '_')
                    filename = self.tables_dir / f"comparison_{model_safe}_{approach}.tex"

                    with open(filename, 'w', encoding='utf-8') as f:
                        f.write(comparison_table)

                    print(f"Saved comparison table: {filename}")

            else:
                print(f"No results found for {model}")

    def print_all_tables(self):
        """Print all LaTeX tables to console"""

        for model in MODELS:
            if model in self.metrics:
                # Individual tables
                for experiment_type in EXPERIMENT_TYPES:
                    if experiment_type in self.metrics[model]:
                        print(f"\n{'='*80}")
                        print(f"LaTeX TABLE FOR {model} - {experiment_type.upper()}")
                        print('='*80)
                        print(self.generate_experiment_type_table(model, experiment_type))

                # Comparison tables
                for approach in APPROACHES:
                    print(f"\n{'='*80}")
                    print(f"COMPARISON TABLE FOR {model} - {approach.upper()} APPROACH")
                    print('='*80)
                    print(self.generate_comparison_table(model, approach))

# ========================== DETAILED ANALYSIS ==========================

def generate_detailed_analysis(metrics: Dict):
    """Generate detailed classification reports"""

    analysis_dir = Path(TABLES_DIR) / "detailed_analysis"
    analysis_dir.mkdir(exist_ok=True)

    # Load raw results again for detailed analysis
    calculator = MetricsCalculator()
    all_results = calculator.load_all_results()

    for model, model_data in all_results.items():
        model_analysis = {}

        for experiment_type, exp_data in model_data.items():
            model_analysis[experiment_type] = {}

            for language, df in exp_data.items():
                if df.empty:
                    continue

                lang_analysis = {}

                for approach in APPROACHES:
                    if f'pred_{approach}_binary' not in df.columns:
                        continue

                    approach_analysis = {}

                    # Binary classification detailed report
                    y_true_bin = ['yes'] * len(df)
                    y_pred_bin = df[f'pred_{approach}_binary'].fillna('no').tolist()
                    y_true_bin_coded = [1 if str(x).lower() == 'yes' else 0 for x in y_true_bin]
                    y_pred_bin_coded = [1 if str(x).lower() == 'yes' else 0 for x in y_pred_bin]

                    # Type classification
                    y_true_type = df['hallucination_type'].fillna('none').tolist()
                    y_pred_type = df[f'pred_{approach}_type'].fillna('none').tolist()

                    # Normalize types
                    y_true_type = calculator.normalize_types(y_true_type)
                    y_pred_type = calculator.normalize_types(y_pred_type)

                    # Level classification
                    y_true_level = df['hallucination_level'].fillna('none').astype(str).str.lower().str.strip().tolist()
                    y_pred_level = df[f'pred_{approach}_degree'].fillna('none').astype(str).str.lower().str.strip().tolist()

                    approach_analysis = {
                        'binary_predictions': {
                            'true_positive': sum(1 for t, p in zip(y_true_bin_coded, y_pred_bin_coded) if t == 1 and p == 1),
                            'false_positive': sum(1 for t, p in zip(y_true_bin_coded, y_pred_bin_coded) if t == 0 and p == 1),
                            'true_negative': sum(1 for t, p in zip(y_true_bin_coded, y_pred_bin_coded) if t == 0 and p == 0),
                            'false_negative': sum(1 for t, p in zip(y_true_bin_coded, y_pred_bin_coded) if t == 1 and p == 0),
                        },
                        'type_distribution': {
                            'ground_truth': pd.Series(y_true_type).value_counts().to_dict(),
                            'predictions': pd.Series(y_pred_type).value_counts().to_dict()
                        },
                        'level_distribution': {
                            'ground_truth': pd.Series(y_true_level).value_counts().to_dict(),
                            'predictions': pd.Series(y_pred_level).value_counts().to_dict()
                        }
                    }

                    lang_analysis[approach] = approach_analysis

                model_analysis[experiment_type][language] = lang_analysis

        # Save detailed analysis
        model_safe = model.replace('/', '_').replace('.', '_')
        analysis_file = analysis_dir / f"detailed_analysis_{model_safe}.json"

        with open(analysis_file, 'w', encoding='utf-8') as f:
            json.dump(model_analysis, f, indent=2, ensure_ascii=False)

        print(f"Saved detailed analysis: {analysis_file}")

# ========================== COMPARISON ANALYSIS ==========================

def generate_audio_text_comparison_analysis(metrics: Dict):
    """Generate detailed analysis comparing audio vs text performance"""

    comparison_dir = Path(TABLES_DIR) / "audio_text_comparison"
    comparison_dir.mkdir(exist_ok=True)

    comparison_results = {}

    for model, model_data in metrics.items():
        if 'audio' not in model_data or 'text' not in model_data:
            continue

        comparison_results[model] = {}

        for language in LANGUAGES:
            if (language not in model_data['audio'] or
                language not in model_data['text']):
                continue

            comparison_results[model][language] = {}

            for approach in APPROACHES:
                if (approach not in model_data['audio'][language] or
                    approach not in model_data['text'][language]):
                    continue

                audio_metrics = model_data['audio'][language][approach]
                text_metrics = model_data['text'][language][approach]

                # Calculate differences
                comparison_results[model][language][approach] = {
                    'binary': {
                        'f1_diff': text_metrics['binary']['f1'] - audio_metrics['binary']['f1'],
                        'acc_diff': text_metrics['binary']['acc'] - audio_metrics['binary']['acc'],
                        'audio': audio_metrics['binary'],
                        'text': text_metrics['binary']
                    },
                    'type': {
                        'f1_diff': text_metrics['type']['f1'] - audio_metrics['type']['f1'],
                        'acc_diff': text_metrics['type']['acc'] - audio_metrics['type']['acc'],
                        'audio': audio_metrics['type'],
                        'text': text_metrics['type']
                    },
                    'level': {
                        'f1_diff': text_metrics['level']['f1'] - audio_metrics['level']['f1'],
                        'acc_diff': text_metrics['level']['acc'] - audio_metrics['level']['acc'],
                        'audio': audio_metrics['level'],
                        'text': text_metrics['level']
                    }
                }

    # Save comparison analysis
    comparison_file = comparison_dir / "audio_text_comparison.json"
    with open(comparison_file, 'w', encoding='utf-8') as f:
        json.dump(comparison_results, f, indent=2, ensure_ascii=False)

    print(f"Saved audio-text comparison: {comparison_file}")

    # Generate summary report
    summary_lines = []
    summary_lines.append("AUDIO vs TEXT COMPARISON SUMMARY")
    summary_lines.append("=" * 50)

    for model, model_data in comparison_results.items():
        summary_lines.append(f"\n{model}")
        summary_lines.append("-" * 30)

        for language, lang_data in model_data.items():
            summary_lines.append(f"\n{language.upper()}")
            summary_lines.append("Task  | Approach | F1 Diff | Acc Diff | Better")
            summary_lines.append("-" * 50)

            for approach, metrics in lang_data.items():
                for task in ['binary', 'type', 'level']:
                    f1_diff = metrics[task]['f1_diff']
                    acc_diff = metrics[task]['acc_diff']
                    better = "Text" if f1_diff > 0 else "Audio" if f1_diff < 0 else "Same"

                    summary_lines.append(f"{task:5} | {approach:8} | {f1_diff:+6.3f} | {acc_diff:+7.3f} | {better}")

    summary_text = "\n".join(summary_lines)

    summary_file = comparison_dir / "comparison_summary.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(summary_text)

    print(f"Saved comparison summary: {summary_file}")
    print("\n" + summary_text)

# ========================== MAIN FUNCTIONS ==========================

def generate_all_tables():
    """Main function to generate all tables and analysis"""

    print("GENERATING CLASSIFICATION RESULTS TABLES")
    print("=" * 60)

    # Calculate metrics
    calculator = MetricsCalculator()
    metrics = calculator.calculate_all_metrics()

    if not metrics:
        print("No metrics calculated. Check if result files exist.")
        return

    # Print summary
    calculator.print_metrics_summary()

    # Generate LaTeX tables
    print(f"\nGenerating LaTeX tables...")
    generator = LaTeXTableGenerator(metrics)
    generator.save_all_tables()

    # Print tables to console
    print(f"\nLaTeX Tables:")
    generator.print_all_tables()

    # Generate detailed analysis
    print(f"\nGenerating detailed analysis...")
    generate_detailed_analysis(metrics)

    # Generate comparison analysis
    print(f"\nGenerating audio-text comparison analysis...")
    generate_audio_text_comparison_analysis(metrics)

    # Save metrics as JSON
    metrics_file = Path(TABLES_DIR) / "all_metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"Saved all metrics: {metrics_file}")

    print(f"\nAll tables and analysis generated in: {TABLES_DIR}")

def quick_metrics_check():
    """Quick function to just see the metrics without generating files"""
    calculator = MetricsCalculator()
    calculator.calculate_all_metrics()
    calculator.print_metrics_summary()

def generate_comparison_only():
    """Generate only the audio-text comparison analysis"""
    calculator = MetricsCalculator()
    metrics = calculator.calculate_all_metrics()

    if metrics:
        generate_audio_text_comparison_analysis(metrics)
    else:
        print("No metrics available for comparison.")

if __name__ == "__main__":
    # Generate all tables and analysis
    generate_all_tables()

    # Or just check metrics quickly
    # quick_metrics_check()

    # Or generate only comparison analysis
    # generate_comparison_only()

GENERATING CLASSIFICATION RESULTS TABLES
Loaded Qwen2.5-Omni-3B - audio - english: 3965 samples
Loaded Qwen2.5-Omni-3B - audio - kazakh: 3977 samples
Loaded Qwen2.5-Omni-3B - audio - russian: 4067 samples
Loaded Qwen2.5-Omni-3B - text - english: 3965 samples
Loaded Qwen2.5-Omni-3B - text - kazakh: 3977 samples
Loaded Qwen2.5-Omni-3B - text - russian: 4067 samples
CLASSIFICATION METRICS SUMMARY

Qwen2.5-Omni-3B
------------------------------------------------------------

AUDIO EXPERIMENTS
----------------------------------------

ENGLISH
Task              | Approach | F1    | Acc   
---------------------------------------------
Binary            | direct   | 0.127 | 0.068
Type              | direct   | 0.300 | 0.547
Level             | direct   | 0.213 | 0.381
Binary            | cot      | 0.111 | 0.059
Type              | cot      | 0.292 | 0.545
Level             | cot      | 0.201 | 0.373

KAZAKH
Task              | Approach | F1    | Acc   
----------------------------------------